In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [7]:
import requests
import json
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

# ==========================================
# CONFIGURACIÓN
# ==========================================

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(HEADERS)


# ==========================================
# URL BUILDER
# ==========================================

def build_listing_url(page):
    return (
        f"https://www.tecnocasa.es/venta/piso/mapa.html/"
        f"pag-{page}?view=41.18051487737182,-2.109104143945018,"
        f"39.64144152264612,-5.319858538475728&zoom=9"
    )


# ==========================================
# EXTRAER LINKS DE UNA PÁGINA
# ==========================================

def get_listing_urls(url):

    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    if not estates_index:
        return []

    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    urls = []

    for estate in estates_data:
        detail_url = estate.get("detail_url")

        if detail_url.startswith("http"):
            urls.append(detail_url)
        else:
            urls.append("https://www.tecnocasa.es" + detail_url)

    return urls


# ==========================================
# EXTRAER DATOS DE UN PISO
# ==========================================

def parse_estate(url):

    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return None

    estate_raw = estate_component.get(":estate")
    estate_raw = estate_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_raw)

    def extract_number(value):
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return value
        if isinstance(value, str):
            match = re.search(r"\d+", value)
            return int(match.group()) if match else None
        return None

    dormitorios = None
    banos = None
    superficie = None

    for item in estate_data.get("data", []):
        label = item.get("label", "").lower()
        value = item.get("valore")

        if "dorm" in label:
            dormitorios = extract_number(value)

        elif "bañ" in label:
            banos = extract_number(value)

        elif "super" in label:
            superficie = extract_number(value)

    base_data = {
        "id": estate_data.get("id"),
        "ciudad": estate_data.get("city", {}).get("title"),
        "precio_raw": estate_data.get("costs", {}).get("price"),
        "superficie_m2": superficie,
        "dormitorios": dormitorios,
        "banos": banos,
        "latitud": estate_data.get("latitude"),
        "longitud": estate_data.get("longitude"),
        "url": url
    }

    features = estate_data.get("features", {})

    for key, value in features.items():
        base_data[f"feature_{key}"] = value

    return base_data


# ==========================================
# LIMPIEZA DE DATOS
# ==========================================

def clean_dataframe(df):

    # Precio → int
    df["precio"] = (
        df["precio_raw"]
        .str.replace(r"[^\d]", "", regex=True)
        .astype("float")
    )

    # Superficie → float
    df["superficie_m2"] = pd.to_numeric(df["superficie_m2"], errors="coerce")

    # Precio por m2
    df["precio_m2"] = df["precio"] / df["superficie_m2"]

    return df


# ==========================================
# SCRAPER PRINCIPAL
# ==========================================

def scrape_all_pages(start=1, end=5, delay=1):

    all_data = []

    for page in range(start, end + 1):

        print(f"\n===== Página {page} =====")

        listing_url = build_listing_url(page)
        estate_urls = get_listing_urls(listing_url)

        print(f"Encontrados {len(estate_urls)} pisos")

        for i, url in enumerate(estate_urls, 1):
            print(f"[{i}/{len(estate_urls)}] {url}")

            try:
                data = parse_estate(url)

                if data:
                    all_data.append(data)

                time.sleep(delay)

            except Exception as e:
                print("Error:", e)

    df = pd.DataFrame(all_data)
    df = clean_dataframe(df)

    return df


# ==========================================
# EJECUCIÓN
# ==========================================

if __name__ == "__main__":

    df = scrape_all_pages(start=1, end=3)

    print("\nResumen DataFrame:")
    print(df.head())

    print("\nColumnas:")
    print(df.columns.tolist())


===== Página 1 =====
Encontrados 15 pisos
[1/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/649574.html
[2/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/648117.html
[3/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/649206.html
[4/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/525984.html
[5/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/625013.html
[6/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/644310.html
[7/15] https://www.tecnocasa.es/venta/piso/madrid/mostoles/631648.html
[8/15] https://www.tecnocasa.es/venta/piso/madrid/valdemorillo/648962.html
[9/15] https://www.tecnocasa.es/venta/piso/madrid/alcorcon/641743.html
[10/15] https://www.tecnocasa.es/venta/piso/madrid/parla/642000.html
[11/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/647649.html
[12/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/646820.html
[13/15] https://www.tecnocasa.es/venta/piso/madrid/madrid/643499.html
[14/15] https://www.tecnocasa.es/venta/piso/mad

In [8]:
df

,id,ciudad,precio_raw,superficie_m2,dormitorios,banos,latitud,longitud,url,feature_id,feature_floor,feature_floors,feature_box,feature_car_places,feature_balconies,feature_terraces,feature_bedrooms,feature_furnitured,feature_air_conditioning,feature_elevator,feature_heating,feature_garden,feature_category,feature_build_year,feature_property_type,feature_concierge,feature_renovation_year,feature_inside_renovation_year,precio,precio_m2
0,649574,Madrid,350.000 €,68,NaN,None,40.408602,-3.705162,https://www.tecnocasa.es/venta/piso/madrid/mad...,649574,1,None,None,None,None,None,None,None,NaN,,NaN,,De época,1900,,,NaN,None,350000.0,5147.058824
1,648117,Madrid,279.000 €,53,2.0,None,40.391002,-3.708712,https://www.tecnocasa.es/venta/piso/madrid/mad...,648117,6,None,None,None,None,None,None,None,Independiente,Sí,centralizada,,Popular,1963,,,NaN,None,279000.0,5264.150943
2,649206,Madrid,437.100 €,160,4.0,None,40.380200,-3.622150,https://www.tecnocasa.es/venta/piso/madrid/mad...,649206,1,None,None,None,None,None,None,None,Independiente (Fan Coil),,Independiente (Gas),,Popular,1965,,,NaN,None,437100.0,2731.875000
3,525984,Madrid,295.000 €,87,3.0,None,40.407002,-3.701732,https://www.tecnocasa.es/venta/piso/madrid/mad...,525984,Baja,None,None,None,None,None,None,None,NaN,,NaN,,Popular,1880,,,NaN,None,295000.0,3390.804598
4,625013,Madrid,200.000 €,42,1.0,None,40.405202,-3.700662,https://www.tecnocasa.es/venta/piso/madrid/mad...,625013,2 (planta media),None,None,None,None,None,None,None,"Independiente (Frío / Calor, A Techo)",,NaN,,Popular,1900,,,NaN,None,200000.0,4761.904762
5,644310,Madrid,150.000 €,26,NaN,None,40.407602,-3.703302,https://www.tecnocasa.es/venta/piso/madrid/mad...,644310,2,None,None,None,None,None,None,None,NaN,,NaN,,Popular,1900,,,NaN,None,150000.0,5769.230769
6,631648,Móstoles,459.000 €,134,3.0,None,40.325702,-3.880092,https://www.tecnocasa.es/venta/piso/madrid/mos...,631648,5 (planta alta),None,None,None,None,None,None,None,Independiente,,Independiente,,Popular,2002,,,NaN,None,459000.0,3425.373134
7,648962,Valdemorillo,270.000 €,120,3.0,None,40.492102,-4.040733,https://www.tecnocasa.es/venta/piso/madrid/val...,648962,1 (planta media),None,None,None,None,None,None,None,NaN,,Independiente,,Popular,1990,,,NaN,None,270000.0,2250.000000
8,641743,Alcorcón,259.000 €,85,3.0,None,40.346402,-3.827622,https://www.tecnocasa.es/venta/piso/madrid/alc...,641743,,None,None,None,None,None,None,None,Independiente,,Independiente,,Media,1973,,,NaN,None,259000.0,3047.058824
9,642000,Parla,179.900 €,75,3.0,None,40.240602,-3.775542,https://www.tecnocasa.es/venta/piso/madrid/par...,642000,,None,None,None,None,None,None,None,Independiente (Frío / Calor),,Independiente (Gas),,Media,1976,,,NaN,None,179900.0,2398.666667


In [4]:
response = requests.get('https://www.tecnocasa.es/venta/piso/madrid/parla/649003.html', timeout=10)
print(BeautifulSoup(response.text, "html.parser"))

<!DOCTYPE html>

<html lang="es-es">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0, maximum-scale=5.1" name="viewport"/>
<meta content="nG5tzP2kVurYVjlc9EdYmNqIXpxExk4G6jsk60pv" name="csrf-token"/>
<title>Piso en venta en Parla - Este, 255.000 €, 80 m2, ref. 649003 - Tecnocasa.es</title>
<meta content="Piso en venta en Parla - Este, 255.000 €, 80 m2, ref. 649003. ¡Contacta con la agencia que lo gestiona para pedir más información! - Tecnocasa.es" name="description">
<meta content="MKautx_6-Jv0E6VAs8uJPkZG1wLRejHrW68L_yWI2Xo" name="google-site-verification">
<meta content="" name="copyright">
<meta content="" name="author"/>
<meta content="General" name="classification"/>
<meta content="General" name="rating"/>
<meta content="Global" name="distribution"/>
<meta content="all,index,follow" name="robots"/>
<link href="https://www.tecnocasa.es/venta/piso/madrid/parla/649003.html" rel="canonical"/>
<meta content="3 days" name="revisit-after"/>
<!-- FB --

In [5]:
response = requests.get('https://www.tecnocasa.es/venta/piso/comunidad-de-madrid/madrid/madrid.html', timeout=10)
print(BeautifulSoup(response.text, "html.parser"))

<!DOCTYPE html>

<html lang="es-es">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0, maximum-scale=5.1" name="viewport"/>
<meta content="GRTHK6uF4BAuncE4HGgCONB94q4lvHNPGMPnnTNQ" name="csrf-token"/>
<title>Piso  en venta en Madrid - Tecnocasa.es</title>
<meta content="Piso  en venta a Madrid: ¿Quieres comprar, vender o alquilar Piso? ¡Descubre las propuestas online las inmobiliarias Tecnocasa.es!" name="description">
<meta content="MKautx_6-Jv0E6VAs8uJPkZG1wLRejHrW68L_yWI2Xo" name="google-site-verification">
<meta content="" name="copyright">
<meta content="" name="author"/>
<meta content="General" name="classification"/>
<meta content="General" name="rating"/>
<meta content="Global" name="distribution"/>
<meta content="all,index,follow" name="robots"/>
<link href="https://www.tecnocasa.es/venta/piso/comunidad-de-madrid/madrid/madrid.html" rel="canonical"/>
<meta content="3 days" name="revisit-after"/>
<!-- FB -->
<meta content="https://www.tecnocas

In [6]:
import requests
import json
from bs4 import BeautifulSoup

url = "https://www.tecnocasa.es/venta/piso/madrid/parla/649003.html"

response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

# 1️⃣ Buscar el componente que contiene el JSON
estate_component = soup.find("estate-show-v2")

# 2️⃣ Extraer el atributo :estate
estate_json_raw = estate_component.get(":estate")

# 3️⃣ Convertir HTML entities a texto normal
estate_json_raw = estate_json_raw.replace("&quot;", '"')

# 4️⃣ Parsear JSON
estate_data = json.loads(estate_json_raw)

# =========================
# EXTRAER INFORMACIÓN
# =========================

data = {
    "ciudad": estate_data["city"]["title"],
    "dormitorios": estate_data["data"][7]["valore"],
    "superficie_m2": estate_data["numeric_surface"],
    "baños": estate_data["bathrooms"],
    "ref": estate_data["id"],
    "planta": estate_data["features"]["floor"],
    "aire_acondicionado": estate_data["features"]["air_conditioning"],
    "ascensor": estate_data["features"]["elevator"],
    "calefaccion": estate_data["features"]["heating"],
    "categoria": estate_data["features"]["category"],
    "año_construccion": estate_data["features"]["build_year"],
    "descripcion": estate_data["description"],
    "precio": estate_data["costs"]["price"],
    "latitud": estate_data["latitude"],
    "longitud": estate_data["longitude"]
}

for k, v in data.items():
    print(f"{k}: {v}")

ciudad: Parla
dormitorios: 3 dormitorios
superficie_m2: 80.00
baños: 2 baños
ref: 649003
planta: 6 (planta alta)
aire_acondicionado: Independiente (Frío / Calor)
ascensor: Sí
calefaccion: Independiente (Gas)
categoria: Media
año_construccion: 2009
descripcion: <p>Agencia inmobiliaria de PARLA - zona PARLA ESTE.<br><br>Vivienda con excelentes vistas al parque del universo a la venta en Parla, zona de Parla Este.<br><br>Situada en pleno centro de Parla Este Junto a los centro de Salud, Farmacia, ahorramas, Parada de tranvía y bus, comercios y servicios de la zona.<br><br>Dispone de 3 dormitorios con armarios empotrados, Cocina de diseño con tendedero cerrado, salón con aire acondicionado, terraza 2 baños. Pintura lisa, suelo de tarima 2 Plazas de garaje y Trastero.<br><br>TECNOCASA, red inmobiliaria líder en el sector con más de 30 años de experiencia, presente con mas 1000 oficinas a nivel nacional y 7 oficinas en Parla.</p>
precio: 255.000 €
latitud: 40.22500174113269
longitud: -3.7551

In [7]:
import requests
import json
import time 
from bs4 import BeautifulSoup

BASE_URL = "https://www.tecnocasa.es"
LISTING_URL = f"https://www.tecnocasa.es/venta/piso/comunidad-de-madrid/madrid/madrid.html/pag-"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(HEADERS)


# ==========================================
# EXTRAER LINKS DE PISOS DE UNA PÁGINA
# ==========================================
def get_listing_urls(url):

    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")

    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')

    estates_data = json.loads(estates_raw)

    urls = [estate["detail_url"] for estate in estates_data]

    return urls


# ==========================================
# EXTRAER DATOS DE UN PISO
# ==========================================
def parse_estate(url):

    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estate_component = soup.find("estate-show-v2")
    estate_raw = estate_component.get(":estate")
    estate_raw = estate_raw.replace("&quot;", '"')

    estate_data = json.loads(estate_raw)

    def get_by_label(data_list, label):
        for item in data_list:
            if item["label"] == label:
                return item["valore"]
        return None

    return {
        "id": estate_data.get("id"),
        "ciudad": estate_data.get("city", {}).get("title"),
        "precio": estate_data.get("costs", {}).get("price"),
        "superficie_m2": estate_data.get("numeric_surface"),
        "dormitorios": get_by_label(estate_data.get("data", []), "Dormitorios"),
        "baños": estate_data.get("bathrooms"),
        "planta": estate_data.get("features", {}).get("floor"),
        "ascensor": estate_data.get("features", {}).get("elevator"),
        "aire_acondicionado": estate_data.get("features", {}).get("air_conditioning"),
        "calefaccion": estate_data.get("features", {}).get("heating"),
        "latitud": estate_data.get("latitude"),
        "longitud": estate_data.get("longitude"),
        "url": url
    }


# ==========================================
# SCRAPING DE UNA PÁGINA COMPLETA
# ==========================================
def scrape_listing_page(listing_url):

    print("Extrayendo URLs...")
    for i in range(1, 41):
        estate_urls = get_listing_urls(listing_url + str(i))

        print(f"Encontrados {len(estate_urls)} pisos")

        all_data = []

        for i, url in enumerate(estate_urls, 1):
            print(f"[{i}/{len(estate_urls)}] Scrapeando: {url}")

            try:
                data = parse_estate(url)
                all_data.append(data)

                time.sleep(0.1)  # polite delay

            except Exception as e:
                print("Error:", e)

        print(all_data)
    return all_data


if __name__ == "__main__":

    data = scrape_listing_page(LISTING_URL)

    print("\nResumen:")
    for estate in data[:3]:
        print(estate)

Extrayendo URLs...
Encontrados 15 pisos
[1/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/649206.html
[2/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/649574.html
[3/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/525984.html
[4/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/625013.html
[5/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/644310.html
[6/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/647649.html
[7/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/646820.html
[8/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/643499.html
[9/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/646154.html
[10/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/646867.html
[11/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madrid/647040.html
[12/15] Scrapeando: https://www.tecnocasa.es/venta/piso/madrid/madr

KeyboardInterrupt: 